In [1]:
import pandas as pd
import numpy as np

# Cargsr desde raw (cada notebook arranca limpio)
df_raw = pd.read_json('../data/raw/streaming_users_dirty.json')
df = df_raw.copy()

n_inicial = len(df)   # para calcular la retención
log = []

def registrar(df, paso, desc):
    log.append({
        'Paso'        : paso,
        'Descripción' : desc,
        'Filas'       : len(df),
        'Nulos'       : df.isnull().sum().sum(),
        'Retención (%)': round(len(df) / n_inicial * 100, 2)
    })

registrar(df, 0, "Dataset original")

Tratamiento de nulos y duplicados

In [2]:
# Filas y usuarios únicos
print("Filas:", len(df))
print("user_id únicos:", df['user_id'].nunique())

Filas: 8160
user_id únicos: 8000


In [3]:
print("Diferencia (filas - ids únicos):", len(df) - df['user_id'].nunique())

Diferencia (filas - ids únicos): 160


In [4]:
print("Filas duplicadas exactas:", df.duplicated().sum())

Filas duplicadas exactas: 126


In [5]:
# Cuántos user_id se repiten?
print("user_id repetidos (en total):", df['user_id'].duplicated().sum())

user_id repetidos (en total): 160


In [6]:
# Saco las exactas en una copia aparte  para explorar
tmp = df.drop_duplicates()
print("Si borro las exactas quedan:", len(tmp), "filas")
print("Pero todavía quedan user_id repetidos:", tmp['user_id'].duplicated().sum())

# mismo usuario, filas que NO son idénticas

Si borro las exactas quedan: 8034 filas
Pero todavía quedan user_id repetidos: 34


In [7]:
# agarro un user_id que quede repetido y miro las dos filas lado a lado
ids_rep = tmp.loc[tmp['user_id'].duplicated(keep=False), 'user_id'].unique()
print("user_id afectados por repetición no-exacta:", len(ids_rep))

uid = ids_rep[0]
tmp[tmp['user_id'] == uid]


user_id afectados por repetición no-exacta: 34


,user_id,age,subscription_plan,monthly_watch_time_mins,country,favorite_genre,last_login_date,customer_support_tickets
59,10059,39,Estándar,2976.6,Brasil,DRAMA,2024-06-22,1
8090,10059,39,Estándar,824.8,Brasil,Drama,2024-06-22,1


In [8]:
# El patrón se repite?
for uid in ids_rep[:4]:
    print("\n---- user_id", uid, "----")
    print(tmp[tmp['user_id'] == uid].to_string())


---- user_id 10059 ----
      user_id  age subscription_plan  monthly_watch_time_mins country favorite_genre last_login_date  customer_support_tickets
59      10059   39          Estándar                   2976.6  Brasil          DRAMA      2024-06-22                         1
8090    10059   39          Estándar                    824.8  Brasil          Drama      2024-06-22                         1

---- user_id 10721 ----
      user_id  age subscription_plan  monthly_watch_time_mins   country favorite_genre last_login_date  customer_support_tickets
721     10721   32          Estándar                    959.0  Colombia     Documental             NaN                         2
8126    10721   32          Estándar                    959.0  Colombia     Documental      2021-09-02                         2

---- user_id 10797 ----
      user_id  age subscription_plan  monthly_watch_time_mins country favorite_genre last_login_date  customer_support_tickets
797     10797   31            

Las diferencias se dan en mayusculas y minusculas y en valores de minutos vistos en un mes (?

Hay dos problemas distintos: Tenemos 126 filas exactas (a eliminar) y 34 user_id repetidos con datos distintos. De estos ultimos, algunos difieren solo en mayúsculas del género(se agruparan al normalizar). Para los que difieran en un número, voy a quedarme con un registro por usuario: el más completo (menos los NaN (user 10721 por ej.)), y optando por last_login_date más reciente, ya que user_id es la clave de usuario y así retengo la fila más informativa. Los conflictos numéricos restantes (0,4%) son inmateriales para los agregados.

Tratamiento de duplicados

In [9]:
# Elimino los duplicados exactos
n_antes = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Duplciados exactos eliminados: {n_antes - len(df)}")   # todavia quedan ids repetidos
registrar(df, 1, "Eliminación de duplicados exactos")

Duplciados exactos eliminados: 126


In [10]:
# Un registro por user_id (el más completo; optando por login más reciente)
n_antes = len(df)
df['_nulos'] = df.isnull().sum(axis=1)
df = (df.sort_values(['_nulos', 'last_login_date'], ascending=[True, False])
        .drop_duplicates(subset='user_id', keep='first')
        .drop(columns='_nulos')
        .reset_index(drop=True))

print(f"user_id duplicados unificados: {n_antes - len(df)}")
print("user_id repetidos que quedan:", df['user_id'].duplicated().sum())   # debería dar 0


user_id duplicados unificados: 34
user_id repetidos que quedan: 0


In [11]:
registrar(df, 2, "Unificación de user_id duplicados")

Normalizar categorias

In [12]:
# Como vienen escritas las categóricas:
for col in ['subscription_plan', 'country', 'favorite_genre']:
    print('===', col, '===')
    print(df[col].value_counts(dropna=False))
    print()

=== subscription_plan ===
subscription_plan
Básico       3386
Estándar     2653
Premium      1481
basico         60
BASICO         52
Basic          52
básico         50
Std            48
Estándar       46
estandar       36
STANDARD       34
Premium        31
PREMIUM        26
Premiun        23
premium        22
Name: count, dtype: int64

=== country ===
country
Chile         1115
Brasil        1107
Uruguay       1102
México        1100
Colombia      1096
Perú          1091
Argentina     1069
colombia        27
méxico          25
uruguay         24
Brazil          21
COL             19
CHL             18
URY             17
PER             16
MEX             16
Chile           16
argentina       16
BRA             15
chile           15
Peru            15
Mexico          15
brasil          13
perú            12
ARG             10
Argentina       10
Name: count, dtype: int64

=== favorite_genre ===
favorite_genre
Comedia        1092
Drama          1082
Thriller       1072
Romance        1

In [13]:
print([repr(x) for x in df['subscription_plan'].unique()])

["'Básico'", "'BASICO'", "'Premium'", "'básico'", "'Estándar '", "'Estándar'", "'Basic'", "'Premiun'", "'estandar'", "'Std'", "'STANDARD'", "'Premium '", "'basico'", "'premium'", "'PREMIUM'"]


Estándar y Premium aparecen dos veces por espacios al final

In [14]:
# Se cuentan cuantos valores cambian si se sacan los espacios
n = (df['subscription_plan'] != df['subscription_plan'].str.strip()).sum()
print("Valores con espacios al borde en subscription_plan:", n)

Valores con espacios al borde en subscription_plan: 77


Las tres variables categóricas vienen con variantes de mayúsculas, idioma, abreviaturas, códigos de país y algunos tipos mal escritos (Premiun, thriler), y espacios en los bordes. 

Decido: .str.strip() primero y después un diccionario map por variable a la forma en español. 

En "favorite_genre" el valor dominante es Crime (inglés) pero el resto de valores de la columna estan en español (Comedia, Drama, Acción). Para seguir la misma linea lo unifico a Crimen. Los None de género no los toco

In [15]:
# Se limpian los espacios antes de mapear
for col in ['subscription_plan', 'country', 'favorite_genre']:
    df[col] = df[col].str.strip()

Mapa plan

In [16]:
mapa_plan = {
    'Básico':'Básico', 'básico':'Básico', 'basico':'Básico', 'BASICO':'Básico', 'Basic':'Básico',
    'Estándar':'Estándar', 'estandar':'Estándar', 'STANDARD':'Estándar', 'Std':'Estándar',
    'Premium':'Premium', 'premium':'Premium', 'PREMIUM':'Premium', 'Premiun':'Premium',
}

Mapa pais

In [17]:
mapa_pais = {
    'Brasil':'Brasil', 'brasil':'Brasil', 'Brazil':'Brasil', 'BRA':'Brasil',
    'Chile':'Chile', 'chile':'Chile', 'CHL':'Chile',
    'México':'México', 'méxico':'México', 'Mexico':'México', 'MEX':'México',
    'Uruguay':'Uruguay', 'uruguay':'Uruguay', 'URY':'Uruguay',
    'Perú':'Perú', 'perú':'Perú', 'Peru':'Perú', 'PER':'Perú',
    'Colombia':'Colombia', 'colombia':'Colombia', 'COL':'Colombia',
    'Argentina':'Argentina', 'argentina':'Argentina', 'ARG':'Argentina',
}

Mapa genero

In [18]:
mapa_genero = {
    'Comedia':'Comedia', 'COMEDIA':'Comedia', 'comedy':'Comedia',
    'Drama':'Drama', 'drama':'Drama', 'DRAMA':'Drama',
    'Documental':'Documental', 'documental':'Documental', 'Documentary':'Documental', 'DOC':'Documental',
    'Romance':'Romance', 'romance':'Romance', 'ROMANCE':'Romance',
    'Thriller':'Thriller', 'THRILLER':'Thriller', 'thriler':'Thriller',
    'Acción':'Acción', 'ACCIÓN':'Acción', 'accion':'Acción', 'Action':'Acción',
    'Crime':'Crimen', 'CRIME':'Crimen', 'crime':'Crimen', 'Crimen':'Crimen',
}

In [19]:
for col, mapa in [('subscription_plan', mapa_plan), ('country', mapa_pais), ('favorite_genre', mapa_genero)]:
    faltan = set(df[col].dropna().unique()) - set(mapa.keys())
    
    print(col, '--> sin mapear:', faltan)

subscription_plan --> sin mapear: set()
country --> sin mapear: set()
favorite_genre --> sin mapear: set()


In [20]:
df['subscription_plan'] = df['subscription_plan'].map(mapa_plan)
df['country']           = df['country'].map(mapa_pais)
df['favorite_genre']    = df['favorite_genre'].map(mapa_genero)

In [21]:
for col in ['subscription_plan', 'country', 'favorite_genre']:
    print(col, '-->', sorted(df[col].dropna().unique()))

subscription_plan --> ['Básico', 'Estándar', 'Premium']
country --> ['Argentina', 'Brasil', 'Chile', 'Colombia', 'México', 'Perú', 'Uruguay']
favorite_genre --> ['Acción', 'Comedia', 'Crimen', 'Documental', 'Drama', 'Romance', 'Thriller']


In [22]:
registrar(df, 3, "Normalización de categóricas")

Tratamiento de valores imposibles

In [23]:
# Como estan las variables numéricas
df[['age', 'monthly_watch_time_mins', 'customer_support_tickets']].describe()

,age,monthly_watch_time_mins,customer_support_tickets
count,8000.000000,7809.000000,8000.000000
mean,34.113500,1112.798130,1.820375
std,14.572797,5363.484471,11.446276
min,-5.000000,-120.000000,-1.000000
25%,25.000000,487.800000,0.000000
50%,33.000000,756.300000,1.000000
75%,42.000000,1044.300000,1.000000
max,150.000000,99999.000000,150.000000


In [24]:
# Revisar valores atipicos
print("age < 0          :", (df['age'] < 0).sum())
print("age > 100        :", (df['age'] > 100).sum())
print("watch < 0        :", (df['monthly_watch_time_mins'] < 0).sum())
print("watch > 43200    :", (df['monthly_watch_time_mins'] > 43200).sum())
print("tickets < 0      :", (df['customer_support_tickets'] < 0).sum())

age < 0          : 21
age > 100        : 53
watch < 0        : 49
watch > 43200    : 31
tickets < 0      : 29


In [25]:
print(df['monthly_watch_time_mins'].value_counts().head(3))

monthly_watch_time_mins
 0.0      164
-120.0     29
-1.0       20
Name: count, dtype: int64


In [26]:
print(df['customer_support_tickets'].value_counts().sort_index(ascending=False).head(5))

customer_support_tickets
150     32
99      35
5       14
4       71
3      280
Name: count, dtype: int64


El dataset tiene muchos valores atipicos.
Se reemplazan por NaN para no borrar la fila para que se traten de forma uniforme

age: válido [13, 100], ponele

monthly_watch_time_mins: negativos y valores > 43200 (el máximo de minutos en un mes), incluyen los centinelas 99999 y 50000 --> NaN

customer_support_tickets: negativos --> NaN.

Los valores 99 y 150 los considero centinelas (frecuencia altísima vs mediana 1) y también --> NaN.

In [27]:
# todo a NaN
df.loc[(df['age'] < 13) | (df['age'] > 100), 'age'] = np.nan

MAX_MIN_MES = 30 * 24 * 60   # 43200 minutos en un mes
df.loc[(df['monthly_watch_time_mins'] < 0) |
       (df['monthly_watch_time_mins'] > MAX_MIN_MES), 'monthly_watch_time_mins'] = np.nan

df.loc[df['customer_support_tickets'] < 0, 'customer_support_tickets'] = np.nan
df.loc[df['customer_support_tickets'].isin([99, 150]), 'customer_support_tickets'] = np.nan

In [28]:
print(df[['age', 'monthly_watch_time_mins', 'customer_support_tickets']].describe())

               age  monthly_watch_time_mins  customer_support_tickets
count  7880.000000              7729.000000               7904.000000
mean     33.712563               794.845465                  0.800481
std      11.542303               496.241873                  0.899081
min      13.000000                 0.000000                  0.000000
25%      25.000000               491.600000                  0.000000
50%      33.000000               757.500000                  1.000000
75%      42.000000              1041.900000                  1.000000
max      80.000000              4193.700000                  5.000000


In [29]:
registrar(df, 4, "Valores imposibles a NaN")

In [30]:
pd.DataFrame(log)

,Paso,Descripción,Filas,Nulos,Retención (%)
0,0,Dataset original,8160,753,100.00
1,1,Eliminación de duplicados exactos,8034,753,98.46
2,2,Unificación de user_id duplicados,8000,743,98.04
3,3,Normalización de categóricas,8000,743,98.04
4,4,Valores imposibles a NaN,8000,1039,98.04


In [31]:
df.isnull().sum().sort_values(ascending=False)

last_login_date             315
monthly_watch_time_mins     271
favorite_genre              237
age                         120
customer_support_tickets     96
user_id                       0
subscription_plan             0
country                       0
dtype: int64

Reemplazo los valores imposibles por `NaN` en vez de borrarlos o rellenarlos. 
A los imposibles los marco como faltante para separar deccisiones distintas.
Por eso los nulos suben, se tratan todos juntos y de forma uniforme más adelante.

In [32]:
# % de faltantes por columna
print((df.isnull().mean() * 100).round(1).sort_values(ascending=False))

last_login_date             3.9
monthly_watch_time_mins     3.4
favorite_genre              3.0
age                         1.5
customer_support_tickets    1.2
user_id                     0.0
subscription_plan           0.0
country                     0.0
dtype: float64


In [33]:
# La ausencia de género se concentra en algún plan o país? (MCAR vs MAR)
print("\nFalta género según plan:")
print(df.assign(falta=df['favorite_genre'].isna()).groupby('subscription_plan')['falta'].mean().round(3))
print("\nFalta login según país:")
print(df.assign(falta=df['last_login_date'].isna()).groupby('country')['falta'].mean().round(3))


Falta género según plan:
subscription_plan
Básico      0.031
Estándar    0.030
Premium     0.027
Name: falta, dtype: float64

Falta login según país:
country
Argentina    0.034
Brasil       0.039
Chile        0.040
Colombia     0.032
México       0.048
Perú         0.042
Uruguay      0.039
Name: falta, dtype: float64


La ausencia de género y de fecha de login se distribuye de forma pareja entre planes y países (≈3–5%), compatible con un mecanismo aleatorio (MCAR), por eso la imputación simple no introduce sesgo de grupo

Tratamiento de faltantes (por tipo de variable)

| Columna | Nulos | Estrategia | Por qué |
|---|---|---|---|
| `last_login_date` | 315 | No imputar | inventar una fecha de acceso falsea el comportamiento; se deja faltante y se parsea en 2.6 |
| `monthly_watch_time_mins` | 271 | mediana | numérica muy asimétrica, la mediana es robusta a la cola/outliers |
| `favorite_genre` | 237 | categoría "Desconocido"  | imputar la moda distorsionaría la distribución de géneros del EDA |
| `age` | 120 | mediana | distribución centrada, la mediana evita arrastrar imposibles previos |
| `customer_support_tickets` | 96 | mediana  | dato de conteo concentrado en 0–1; la mediana respeta esa concentración |

In [34]:
# numéricas --> mediana
for col in ['monthly_watch_time_mins', 'age', 'customer_support_tickets']:
    df[col] = df[col].fillna(df[col].median())

# categórica --> etiqueta explícita
df['favorite_genre'] = df['favorite_genre'].fillna('Desconocido')

# last_login_date: NO se imputa

# age y tickets son enteros conceptualmente --> los devuelvo a int
df['age'] = df['age'].round().astype(int)
df['customer_support_tickets'] = df['customer_support_tickets'].round().astype(int)

print(df.isnull().sum().sort_values(ascending=False))
registrar(df, 5, "Imputación de faltantes")

last_login_date             315
user_id                       0
subscription_plan             0
age                           0
monthly_watch_time_mins       0
country                       0
favorite_genre                0
customer_support_tickets      0
dtype: int64


In [35]:
pd.DataFrame(log)

,Paso,Descripción,Filas,Nulos,Retención (%)
0,0,Dataset original,8160,753,100.00
1,1,Eliminación de duplicados exactos,8034,753,98.46
2,2,Unificación de user_id duplicados,8000,743,98.04
3,3,Normalización de categóricas,8000,743,98.04
4,4,Valores imposibles a NaN,8000,1039,98.04
5,5,Imputación de faltantes,8000,315,98.04


In [36]:
df['last_login_date'].dropna().sample(15, random_state=1).tolist()

['2021-02-14',
 '2023-09-14',
 '2018-03-02',
 '2022-04-04',
 '2020-10-09',
 '2018-06-09',
 '2021-07-15',
 '2024-10-12',
 '2022-08-10',
 '2020-03-05',
 '2018-06-04',
 '2023-01-28',
 '2019-03-03',
 '2021-01-29',
 '2025-01-04']

In [37]:
import re
s = df['last_login_date'].dropna().astype(str)
def patron(x):
    if re.fullmatch(r'\d{4}-\d{2}-\d{2}', x): return 'ISO YYYY-MM-DD'
    if re.fullmatch(r'\d{4}/\d{2}/\d{2}', x): return 'slash YYYY/MM/DD'
    if re.fullmatch(r'\d{2}-\d{2}-\d{4}', x): return 'DD-MM-YYYY'
    return 'OTRO: ' + x
print(s.map(patron).value_counts())

last_login_date
ISO YYYY-MM-DD         7277
DD-MM-YYYY              261
slash YYYY/MM/DD        133
OTRO: not_available      14
Name: count, dtype: int64


Parseo de fechas
* `last_login_date` viene en tres formatos válidos (ISO, `YYYY/MM/DD`, `DD-MM-YYYY`) y un texto `not_available` que es un faltante disfrazado.
Parseo cada formato con su patrón explícito (para no confundir día con mes en `DD-MM-YYYY`), lo que no matchea ningún formato — incluido `not_available` — queda como `NaT`. No imputo fechas: dejo el faltante.

In [38]:
from datetime import datetime

def parse_fecha(x):
    if pd.isna(x):
        return pd.NaT
    if isinstance(x, (pd.Timestamp, datetime)):
        return x
    x = str(x).strip()
    for fmt in ('%Y-%m-%d', '%Y/%m/%d', '%d-%m-%Y'):
        try:
            return pd.to_datetime(x, format=fmt)
        except (ValueError, TypeError):
            continue
    return pd.NaT

df['last_login_date'] = df['last_login_date'].apply(parse_fecha)

print("dtype:", df['last_login_date'].dtype)
print("Fechas faltantes (NaT) tras parsear:", df['last_login_date'].isna().sum())
print("Rango:", df['last_login_date'].min(), "→", df['last_login_date'].max())

dtype: datetime64[us]
Fechas faltantes (NaT) tras parsear: 457
Rango: 2018-01-01 00:00:00 → 2029-01-01 00:00:00


In [39]:
registrar(df, 6, "Parseo de last_login_date")

In [40]:
# un login no puede estar en el futuro: lo trato como imposible, igual que las edades absurdas
hoy = pd.Timestamp('2026-06-27')
n_fut = (df['last_login_date'] > hoy).sum()
df.loc[df['last_login_date'] > hoy, 'last_login_date'] = pd.NaT

print(f"Fechas futuras pasadas a NaT: {n_fut}")
print("NaT totales ahora:", df['last_login_date'].isna().sum())

Fechas futuras pasadas a NaT: 15
NaT totales ahora: 472


In [41]:
registrar(df, 7, "Fechas futuras imposibles a NaT")

Tras parsear detecto 15 fechas en 2029 (futuras respecto de hoy). Un último login no puede estar en el futuro: las trato como imposibles y las paso a NaT, igual que los valores fuera de rango

In [42]:
# antigüedad del último acceso en días: variable numérica extra, me va a servir para el PCA
df['dias_desde_login'] = (hoy - df['last_login_date']).dt.days

print(df['dias_desde_login'].describe())
print("\nNaN en dias_desde_login:", df['dias_desde_login'].isna().sum())

count    7528.000000
mean     1607.880579
std       845.193590
min       178.000000
25%       865.000000
50%      1594.000000
75%      2348.000000
max      3099.000000
Name: dias_desde_login, dtype: float64

NaN en dias_desde_login: 472


In [43]:
registrar(df, 8, "Derivación de dias_desde_login")

In [44]:
# dataset limpio
df.to_csv('../data/processed/streaming_users_clean.csv', index=False)
print("Guardado: data/processed/streaming_users_clean.csv  | filas:", len(df), "| cols:", df.shape[1])

Guardado: data/processed/streaming_users_clean.csv  | filas: 8000 | cols: 9


In [45]:
# log
log_df = pd.DataFrame(log)
log_df.to_csv('../logs/pipeline_log.csv', index=False)

print("\nGuardado: logs/pipeline_log.csv")
print(log_df.to_string(index=False))


Guardado: logs/pipeline_log.csv
 Paso                       Descripción  Filas  Nulos  Retención (%)
    0                  Dataset original   8160    753         100.00
    1 Eliminación de duplicados exactos   8034    753          98.46
    2 Unificación de user_id duplicados   8000    743          98.04
    3      Normalización de categóricas   8000    743          98.04
    4          Valores imposibles a NaN   8000   1039          98.04
    5           Imputación de faltantes   8000    315          98.04
    6         Parseo de last_login_date   8000    457          98.04
    7   Fechas futuras imposibles a NaT   8000    472          98.04
    8    Derivación de dias_desde_login   8000    944          98.04


Síntesis de la limpieza

* Partí de **8160 registros** y terminé con **8000** tras eliminar **126 duplicados exactos** y unificar **34 user_id repetidos** (me quedé con el registro más completo de cada usuario). Normalicé `subscription_plan`, `country` y `favorite_genre`, que venían en varias formas (mayúsculas, idioma, abreviaturas, typos y espacios), a su forma canónica, sin generar faltantes nuevos.

* Marqué como faltante todo valor imposible —edades fuera de [13, 100], `watch_time` mayor a 43200 minutos (el máximo de un mes, que incluye los centinelas 99999 y 50000) y negativos, tickets negativos y los centinelas 99/150— lo que elevó los nulos a 1039. Después imputé las numéricas con la **mediana** y el género ausente con la categoría **"Desconocido"**, bajando los faltantes a 315. Parsear `last_login_date` (tres formatos mezclados) mostró ~142 fechas no válidas, y también encontré **15 fechas futuras (2029)** que pasé a NaT por imposibles.

* Las fechas faltantes no se imputan: quedan **472 registros sin `last_login_date`**. Al derivar `dias_desde_login` esos mismos 472 huecos se reflejan en la nueva columna, por lo que el total de nulos final es **944**, concentrados en las dos variables de fecha (la original y la derivada) y compartiendo exactamente las mismas filas. En el notebook 04 veré si se imputan o se excluyen para el PCA.

Retención estructural final: 98,04%** — no se descartó ninguna fila válida.